# BioNeuronAI on Google Colab

**Repo:** https://github.com/kyle0527/BioNeuronai  
**Goal:** Python **3.13** (micromamba) + **GPU torch** + smoke (`status`) + optional short training.

### Before you run
1. **Runtime → Change runtime type → GPU** (e.g. T4)
2. Reconnect, then run cells top to bottom

Paper trading long-runs: use **local machine**, not Colab.  
Docs: `docs/manuals/21_COLAB.md`

## 0. Diagnose system runtime (often Python 3.12)

In [ ]:
import sys
print("system python", sys.version)
try:
    import torch
    print("system torch", torch.__version__, "cuda", torch.cuda.is_available())
except Exception as e:
    print("system torch", e)
!nvidia-smi -L 2>/dev/null || echo "no GPU — set Runtime to GPU and reconnect"

## 1. Clone repository (public)

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/kyle0527/BioNeuronai.git"
ROOT = Path("/content/BioNeuronai")

if not (ROOT / "main.py").exists():
    !git clone {REPO_URL} {ROOT}
else:
    print("already cloned; pulling…")
    !git -C {ROOT} pull --ff-only || true

os.chdir(ROOT)
print("cwd", os.getcwd())
!git -C {ROOT} log -1 --oneline

## 2. Setup Python 3.13 env (micromamba) + deps + GPU torch

Takes several minutes. Uses `tools/colab/setup_colab.sh`.

In [ ]:
import os
from pathlib import Path

ROOT = Path("/content/BioNeuronai")
os.chdir(ROOT)
os.environ["BIONEURONAI_ROOT"] = str(ROOT)
os.environ["MAMBA_ROOT_PREFIX"] = "/content/micromamba"

!chmod +x tools/colab/setup_colab.sh
!bash tools/colab/setup_colab.sh

## 3. Smoke with env Python 3.13

Always call the **env** interpreter (not system 3.12).

In [ ]:
from pathlib import Path

PY = Path("/content/micromamba/envs/bioneuronai/bin/python")
assert PY.exists(), f"missing {PY}; re-run setup cell"

!{PY} -c "import sys; print(sys.version)"
!{PY} -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
!{PY} -c "import bioneuronai; print('bioneuronai OK')"
!{PY} main.py status

## 4. Optional — mount Drive for data / checkpoints

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# RUN_DIR = "/content/drive/MyDrive/bioneuronai_runs"
# !mkdir -p {RUN_DIR}
print("Uncomment above to mount Google Drive")

## 5. Optional — unified_trainer dry-run

Requires a small signal JSONL (upload or generate on a machine with history data).  
See `docs/manuals/13_CLOUD_TRAINING_RUNBOOK.md`.

In [ ]:
# SIGNAL = "/content/drive/MyDrive/bioneuronai_data/unified_v2_training.jsonl"
# OUT = "/content/drive/MyDrive/bioneuronai_runs/dryrun"
# PY = "/content/micromamba/envs/bioneuronai/bin/python"
# !{PY} -m nlp.training.unified_trainer \
#   --sig-only \
#   --signal-data {SIGNAL} \
#   --max-signal-samples 4 \
#   --epochs 1 \
#   --batch 2 \
#   --grad-accum 1 \
#   --output {OUT} \
#   --no-save
print("Uncomment and set SIGNAL path when data is ready")

## Notes

- After runtime restart: re-run clone (if needed) + setup, or only activate env python path.
- Do **not** promote `config/active_model.json` from experimental Colab runs without local validation.
- Secrets: use Colab Secrets / userdata for API keys; never commit `.env`.